In [2]:
## Step 1. 준비

import json
import os
import re
import time
from datetime import datetime

import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
ollama_url = os.getenv("OLLAMA_URL")

print("OpenAI key :", "있음" if openai_api_key else "없음 — .env 를 확인하세요")
print("Ollama URL :", ollama_url)

# Ollama 가 실제로 떠 있는지, 어떤 모델이 받아져 있는지 지금 확인합니다.
# 40회 호출을 다 돌린 뒤에 "연결 안 됨"을 알게 되면 시간만 버립니다.
try:
    probe = OpenAI(api_key="ollama", base_url=ollama_url)
    installed = sorted(m.id for m in probe.models.list())
    print("로컬 모델  :", ", ".join(installed) or "없음 — ollama pull 로 받으세요")
except Exception as e:
    print("로컬 모델  : 연결 실패 —", type(e).__name__)
    print("             Ollama 가 떠 있는지 터미널에서 'ollama list' 로 확인하세요")

OpenAI key : 있음
Ollama URL : http://localhost:11434/v1
로컬 모델  : llama3.2:3b, llama3.2:latest, qwen2.5:3b


In [3]:
advisories = json.loads(open("advisories_sample.json", encoding="utf-8").read())

print(f"공지 {len(advisories)}건\n")
for a in advisories:
    print(f"{a['no']:2}. {a['cve_id']:16} {a['label']:22} {len(a['advisory']):5}자")

공지 10건

 1. CVE-2021-44228   Apache Log4j2           2500자
 2. CVE-2022-22965   Spring Framework        1830자
 3. CVE-2017-5638    Apache Struts 2         1726자
 4. CVE-2014-6271    GNU Bash                1011자
 5. CVE-2023-49105   ownCloud                 383자
 6. CVE-2026-48710   Starlette               2500자
 7. CVE-2026-59822   LiteLLM                 1752자
 8. CVE-2026-75650   Adobe Commerce           350자
 9. CVE-2026-49869   Kestra                   772자
10. CVE-2026-73570   Zimbra Collaboration     418자


In [4]:
# 공지 하나가 어떻게 생겼는지 눈으로 확인
print(advisories[4]["cve_id"], "|", advisories[4]["label"])
print()
print(advisories[4]["advisory"])

CVE-2023-49105 | ownCloud

An issue was discovered in ownCloud owncloud/core before 10.13.1. An attacker can access, modify, or delete any file without authentication if the username of a victim is known, and the victim has no signing-key configured. This occurs because pre-signed URLs can be accepted even when no signing-key is configured for the owner of the files. The earliest affected version is 10.6.0.


In [5]:
SYSTEM_PROMPT = """당신은 보안 공지를 요약하는 분석가입니다.

아래 형식으로 **정확히 네 줄**만 답하세요. 다른 말을 덧붙이지 마세요.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있는지 한 문장>
조치: <올려야 할 버전. 공지에 버전이 없으면 정확히 "공지에 없음">
긴급도: <긴급|높음|보통|낮음 중 하나>

규칙
- 공지에 적힌 내용만 쓰세요. 공지에 없는 버전·날짜·수치를 지어내지 마세요.
- 버전이 공지에 없으면 반드시 "공지에 없음"이라고 쓰세요. 추측하지 마세요.
- 한국어로 답하세요."""


def build_messages(advisory_text):
    """공지 하나를 messages 로 만든다."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"다음 보안 공지를 요약하세요.\n\n{advisory_text}"},
    ]


print(SYSTEM_PROMPT)

당신은 보안 공지를 요약하는 분석가입니다.

아래 형식으로 **정확히 네 줄**만 답하세요. 다른 말을 덧붙이지 마세요.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있는지 한 문장>
조치: <올려야 할 버전. 공지에 버전이 없으면 정확히 "공지에 없음">
긴급도: <긴급|높음|보통|낮음 중 하나>

규칙
- 공지에 적힌 내용만 쓰세요. 공지에 없는 버전·날짜·수치를 지어내지 마세요.
- 버전이 공지에 없으면 반드시 "공지에 없음"이라고 쓰세요. 추측하지 마세요.
- 한국어로 답하세요.


In [6]:
openai_client = OpenAI()
ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)   # 키는 아무 값이나 됩니다

# 이름: (클라이언트, 모델 ID, max_tokens)
# 모델을 더 비교하고 싶으면 이 딕셔너리에 한 줄만 넣으면 됩니다.
MODELS = {
    "gpt-4.1-mini": (openai_client, "gpt-4.1-mini", 300),
    "gpt-4.1-nano": (openai_client, "gpt-4.1-nano", 300),
    "qwen2.5:3b":   (ollama_client, "qwen2.5:3b",   300),
    "llama3.2":     (ollama_client, "llama3.2",     300),
}

print("비교할 모델", len(MODELS), "개")
for name, (_, model_id, mt) in MODELS.items():
    where = "로컬" if _ is ollama_client else "클라우드"
    print(f"  {name:16} {where:5} -> {model_id:16} max_tokens={mt}")

비교할 모델 4 개
  gpt-4.1-mini     클라우드  -> gpt-4.1-mini     max_tokens=300
  gpt-4.1-nano     클라우드  -> gpt-4.1-nano     max_tokens=300
  qwen2.5:3b       로컬    -> qwen2.5:3b       max_tokens=300
  llama3.2         로컬    -> llama3.2         max_tokens=300


In [7]:
ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)

In [8]:
encoding = tiktoken.get_encoding("o200k_base")   # gpt-4.1 계열


def count_tokens(messages):
    return sum(len(encoding.encode(m["content"])) for m in messages)


total_in = 0
print(f"{'번호':>4} {'CVE':16} {'입력 토큰':>9}")
for a in advisories:
    n = count_tokens(build_messages(a["advisory"]))
    total_in += n
    print(f"{a['no']:>4} {a['cve_id']:16} {n:>9,}")

print(f"\n공지 10건 합계: {total_in:,} 토큰 (모델 1개 기준)")
print(f"모델 {len(MODELS)}개면 약 {total_in * len(MODELS):,} 토큰")

  번호 CVE                  입력 토큰
   1 CVE-2021-44228         789
   2 CVE-2022-22965         633
   3 CVE-2017-5638          573
   4 CVE-2014-6271          370
   5 CVE-2023-49105         274
   6 CVE-2026-48710         672
   7 CVE-2026-59822         560
   8 CVE-2026-75650         250
   9 CVE-2026-49869         350
  10 CVE-2026-73570         269

공지 10건 합계: 4,740 토큰 (모델 1개 기준)
모델 4개면 약 18,960 토큰
